In [ ]:
# 3 - modeling

# The purpose of this notebook is to use optuna to do hyperparameter optimization for the 
# current model. We also look at a model trained on standards and tested on standards instead of the real paint
# to see how that performs. The model trained on standards test on standards did do better which
# matches intuition

#
# insight: the best model optuna gave was good, we are going to use the optuna model traine donly on standards.
# After visually inspecting the worst errors the model is making, they actually make sense based on the gel site image visually more than the human rated scores.
# feeling that the model is capturing what it should at this point. Also, optuna model trained with only standards did pretty much the same as model
# trained with standards + low-std real paint, so lets go with the model trained only on the standards.
# The best model code will be found at the bottom of the notebook. 
#
# 

In [ ]:
# imports

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

import sys
import os
import importlib
import numpy as np

sys.path.append(os.path.abspath(".."))
from src.data import load_images_from_metadata, filter_images

In [4]:
# load all of the data into metadata csv file including the new data (data_new_2))
from src.build_metadata import load_metadata
import importlib
import src.build_metadata as build_metadata

importlib.reload(build_metadata)
build_metadata.load_metadata(
    data_base_dir="../data",
    new_data_base_dir="../data_new",
    new_data_2_base_dir="../data_new_2",
    output_file="metadata.csv",
)

Applied human ratings to 851 DD rows.
Saved metadata to ../data\metadata.csv
Combined records: 1345


,LevelingScore,Person,DateCollected,ImageType,GSCamera,PanelID,State,FilePath
0,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (1...
1,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (2...
2,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (3...
3,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (4...
4,1.3,None,None,DD,None,25-130,raw,../data\raw\1\Leveling-1_GS-25-130_0degrees (5...
...,...,...,...,...,...,...,...,...
1340,5.8,Brooke,7.21.26,TEST,2BDR-9F02,25-143,flat,../data_test\{leveling-5.8}_P{Brooke}_D{7.21.2...
1341,6.3,Brooke,7.21.26,TEST,2BDR-9F02,25-141,flat,../data_test\{leveling-6.3}_P{Brooke}_D{7.21.2...
1342,6.8,Brooke,7.21.26,TEST,2BDR-9F02,25-104,flat,../data_test\{leveling-6.8}_P{Brooke}_D{7.21.2...
1343,8.0,Brooke,7.21.26,TEST,2BDR-9F02,25-125,flat,../data_test\{leveling-8.0}_P{Brooke}_D{7.21.2...


In [5]:
# get the data
data = load_images_from_metadata("../data/metadata.csv", crop_fraction = .4, use_cv2=True)

data_standards = filter_images(data, ImageType = "STD", State = "flat", DateCollected = ["6.2.2026", "6.3.2026", "6.5.2026", "6.8.2026"])
data_real_paint = filter_images(data, ImageType = "DD", State = "flat")

Loaded 1345 images (cropping=ON, backend=cv2)
Filtered down to 177 images
Filtered down to 430 images


In [6]:
# Wavelet-style model setup for Optuna

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

SEED = 42


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Keep kernels deterministic across reruns.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


set_global_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def prepare_tensors(data, img_size=(64, 64)):
    X, y, groups = [], [], []
    for d in data:
        img = d["image"].astype(np.float32) / 255.0
        if img.ndim != 2:
            raise ValueError(f"Expected grayscale 2D image, got shape {img.shape}")

        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
        img = torch.nn.functional.interpolate(
            img.unsqueeze(0),
            size=img_size,
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        X.append(img)
        y.append(float(d["LevelingScore"]))
        groups.append(d.get("PanelID", None))

    return X, np.array(y, dtype=np.float32), np.array(groups)


class TensorListDataset(Dataset):
    def __init__(self, X_list, y_array, transform=None):
        self.X_list = X_list
        self.y_array = y_array
        self.transform = transform

    def __len__(self):
        return len(self.X_list)

    def __getitem__(self, idx):
        x = self.X_list[idx].clone()
        if self.transform is not None:
            x = self.transform(x)
        y = torch.tensor(self.y_array[idx], dtype=torch.float32)
        return x, y


def split_indices_group_safe(groups, val_fraction=0.2, seed=42):
    groups = np.array(groups)
    effective_groups = np.array([
        g if isinstance(g, str) and g.strip() != "" else f"NO_PANEL_{i}"
        for i, g in enumerate(groups)
    ])

    unique_groups = np.unique(effective_groups)
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_groups)

    n_val_groups = max(1, int(round(val_fraction * len(unique_groups))))
    val_group_set = set(unique_groups[:n_val_groups])

    train_idx = np.array([i for i, g in enumerate(effective_groups) if g not in val_group_set])
    val_idx = np.array([i for i, g in enumerate(effective_groups) if g in val_group_set])
    return train_idx, val_idx


class WaveletCNNRegressor(nn.Module):
    def __init__(self, num_blocks=5, base_channels=16, hidden_dim=128, dropout=0.5, use_batchnorm=True):
        super().__init__()
        self.pool = nn.AvgPool2d(2)

        layers = []
        in_channels = 1
        channels = base_channels
        for i in range(num_blocks):
            layers.append(nn.Conv2d(in_channels, channels, 3, padding=1))
            if use_batchnorm:
                layers.append(nn.BatchNorm2d(channels))
            layers.extend([
                nn.ReLU(),
                nn.Conv2d(channels, channels, 3, padding=1),
                nn.ReLU(),
            ])
            if i < num_blocks - 1:
                layers.append(nn.MaxPool2d(2))

            in_channels = channels
            channels *= 2

        layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        self.features = nn.Sequential(*layers)

        self.regressor = nn.Sequential(
            nn.Linear(in_channels + 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        x_low1 = self.pool(x)
        x_low2 = self.pool(x_low1)

        m0 = x.mean(dim=[2, 3])
        m1 = x_low1.mean(dim=[2, 3])
        m2 = x_low2.mean(dim=[2, 3])
        wavelet_feats = torch.cat([m0, m1, m2], dim=1)

        feats = self.features(x)
        feats = feats.view(feats.size(0), -1)

        combined = torch.cat([feats, wavelet_feats], dim=1)
        return self.regressor(combined).squeeze(1)


def make_transforms(img_size, rotation_deg, crop_scale):
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(rotation_deg),
        transforms.RandomResizedCrop(
            img_size,
            scale=(crop_scale, 1.0),
            antialias=True,
        ),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    val_tf = transforms.Compose([
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])

    return train_tf, val_tf


def weighted_sampler_from_y(y, seed=None):
    y_int = np.rint(y).astype(int)
    unique, counts = np.unique(y_int, return_counts=True)
    freq = {k: c for k, c in zip(unique, counts)}
    weights = np.array([1.0 / freq[v] for v in y_int], dtype=np.float32)

    generator = None
    if seed is not None:
        generator = torch.Generator()
        generator.manual_seed(seed)

    return WeightedRandomSampler(
        torch.tensor(weights),
        num_samples=len(weights),
        replacement=True,
        generator=generator,
    )


def build_loaders_weighted(X_list, y_array, train_idx, val_idx, batch_size, train_tf, val_tf, seed=None):
    X_train = [X_list[i] for i in train_idx]
    y_train = y_array[train_idx]
    X_val = [X_list[i] for i in val_idx]
    y_val = y_array[val_idx]

    train_ds = TensorListDataset(X_train, y_train, transform=train_tf)
    val_ds = TensorListDataset(X_val, y_val, transform=val_tf)

    sampler = weighted_sampler_from_y(y_train, seed=seed)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader


X_ref, y_ref, groups_ref = prepare_tensors(data_standards, img_size=(64, 64))
train_idx, val_idx = split_indices_group_safe(groups_ref, val_fraction=0.2, seed=SEED)

train_groups = set(groups_ref[train_idx].tolist())
val_groups = set(groups_ref[val_idx].tolist())
group_overlap = train_groups.intersection(val_groups)

print(f"Using device: {DEVICE}")
print(f"Standards total: {len(data_standards)} | train: {len(train_idx)} | val: {len(val_idx)}")
print(f"Real paint test set: {len(data_real_paint)}")
print(f"Leakage check (train/val group overlap): {len(group_overlap)}")

Using device: cpu
Standards total: 177 | train: 142 | val: 35
Real paint test set: 430
Leakage check (train/val group overlap): 0


In [7]:
import optuna


def make_optimizer(name, params, lr, weight_decay):
    if name == "Adam":
        return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "RMSprop":
        return optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)


def make_loss(name):
    if name == "L1":
        return nn.L1Loss()
    if name == "MSE":
        return nn.MSELoss()
    return nn.SmoothL1Loss(beta=0.5)


def evaluate_mae(model, loader, device):
    model.eval()
    errs = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            preds = model(images)
            errs.append(torch.abs(preds - labels).detach().cpu())

    if not errs:
        return float("inf")

    return torch.cat(errs).mean().item()


EPOCHS = 24
N_TRIALS = 60
TIMEOUT_SECONDS = 45 * 60


def objective(trial):
    trial_seed = SEED + trial.number
    set_global_seed(trial_seed)

    img_size = trial.suggest_categorical("img_size", [64, 92])
    num_blocks = trial.suggest_categorical("num_blocks", [4, 5])
    base_channels = trial.suggest_categorical("base_channels", [12, 16, 24])
    hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.2, 0.5)
    use_batchnorm = trial.suggest_categorical("use_batchnorm", [True])

    batch_size = trial.suggest_categorical("batch_size", [16, 32])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])
    lr = trial.suggest_float("lr", 3e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-8, 1e-5, log=True)

    rotation_deg = trial.suggest_int("rotation_deg", 0, 8)
    crop_scale = trial.suggest_float("crop_scale", 0.93, 1.0)
    loss_name = trial.suggest_categorical("loss", ["L1", "SmoothL1"])

    X_std, y_std, _ = prepare_tensors(data_standards, img_size=(img_size, img_size))
    train_tf, val_tf = make_transforms(img_size, rotation_deg, crop_scale)
    train_loader, val_loader = build_loaders_weighted(
        X_std,
        y_std,
        train_idx,
        val_idx,
        batch_size,
        train_tf,
        val_tf,
        seed=trial_seed,
    )

    model = WaveletCNNRegressor(
        num_blocks=num_blocks,
        base_channels=base_channels,
        hidden_dim=hidden_dim,
        dropout=dropout,
        use_batchnorm=use_batchnorm,
    ).to(DEVICE)

    optimizer = make_optimizer(optimizer_name, model.parameters(), lr, weight_decay)
    criterion = make_loss(loss_name)

    best_val = float("inf")

    for epoch in range(EPOCHS):
        model.train()
        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()
            preds = model(images)
            loss = criterion(preds, labels)
            loss.backward()
            optimizer.step()

        val_mae = evaluate_mae(model, val_loader, DEVICE)
        best_val = min(best_val, val_mae)

        trial.report(val_mae, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val


set_global_seed(SEED)
sampler = optuna.samplers.TPESampler(seed=SEED)
pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=6)

study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SECONDS)

print("Optimization finished.")
print(f"Best validation MAE: {study.best_value:.4f}")

c:\Users\bxb370\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-30 11:40:54,246] A new study created in memory with name: no-name-155d5d18-59cd-4581-a23c-593d2f4749c9
[I 2026-07-30 11:42:04,739] Trial 0 finished with value: 0.6231434941291809 and parameters: {'img_size': 92, 'num_blocks': 4, 'base_channels': 12, 'hidden_dim': 64, 'dropout': 0.20617534828874073, 'use_batchnorm': True, 'batch_size': 16, 'optimizer': 'Adam', 'lr': 0.0004576418837415782, 'weight_decay': 8.179499475211672e-08, 'rotation_deg': 4, 'crop_scale': 0.9602361513049481, 'loss': 'SmoothL1'}. Best is trial 0 with value: 0.6231434941291809.
[I 2026-07-30 11:43:23,175] Trial 1 finished with value: 0.4067451059818268 and parameters: {'img_size': 92, 'num_blocks': 5, 'base_channels': 12, 'hidden_dim

Optimization finished.
Best validation MAE: 0.4067


In [ ]:
# best optuna paramaters: 

# Trial 1 finished with value: 0.4067451059818268 and parameters: {'img_size': 92, 'num_blocks': 5, 'base_channels': 12, 'hidden_dim': 256, 'dropout': 0.2511572371061875, 'use_batchnorm': True, 'batch_size': 32, 'optimizer': 'Adam', 'lr': 0.0006049716507542575, 'weight_decay': 1.963434157293331e-08, 'rotation_deg': 6, 'crop_scale': 0.9608106745617722, 'loss': 'SmoothL1'}. Best is trial 1 with value: 0.4067451059818268.



In [ ]:
# now lets look at optuna architecture just tested on standards

In [8]:
# standards-only training/eval using Optuna best hyperparameters

import numpy as np
import torch
import pandas as pd
from sklearn.model_selection import train_test_split

if "study" not in globals() or len(study.trials) == 0:
    raise RuntimeError("Run the Optuna optimization cell first so study.best_trial is available.")

best_p = study.best_trial.params
set_global_seed(SEED)

img_size = int(best_p["img_size"])
batch_size = int(best_p["batch_size"])
rotation_deg = int(best_p["rotation_deg"])
crop_scale = float(best_p["crop_scale"])
epochs = 50

# Prepare tensors with Optuna-selected image size
X_all, y_all, groups_all = prepare_tensors(data_standards, img_size=(img_size, img_size))

# Random stratified split by label (image-level split, panels can appear in both sets)
all_idx = np.arange(len(y_all))
train_idx_local, val_idx_local = train_test_split(
    all_idx,
    test_size=0.2,
    random_state=SEED,
    stratify=y_all,
    shuffle=True,
)

groups_val = groups_all[val_idx_local]

train_tf, val_tf = make_transforms(
    img_size=img_size,
    rotation_deg=rotation_deg,
    crop_scale=crop_scale,
)

train_loader, val_loader = build_loaders_weighted(
    X_all,
    y_all,
    train_idx_local,
    val_idx_local,
    batch_size,
    train_tf,
    val_tf,
    seed=SEED,
)

model_std = WaveletCNNRegressor(
    num_blocks=int(best_p["num_blocks"]),
    base_channels=int(best_p["base_channels"]),
    hidden_dim=int(best_p["hidden_dim"]),
    dropout=float(best_p["dropout"]),
    use_batchnorm=bool(best_p["use_batchnorm"]),
).to(DEVICE)

optimizer_std = make_optimizer(
    best_p["optimizer"],
    model_std.parameters(),
    float(best_p["lr"]),
    float(best_p["weight_decay"]),
)
criterion_std = make_loss(best_p["loss"])

print("=" * 60)
print("TRAIN ON STANDARDS-TRAIN, TEST ON STRATIFIED STANDARDS-VAL")
print("USING OPTUNA BEST PARAMS")
print("=" * 60)
print(f"Standards total: {len(X_all)}")
print(f"Train images: {len(train_idx_local)} | Val images: {len(val_idx_local)}")
print("Split type: random stratified by label (not panel-safe)")
print("Best params:")
for k, v in best_p.items():
    print(f"  {k}: {v}")

for epoch in range(epochs):
    model_std.train()
    running_loss = 0.0
    n_seen = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_std.zero_grad()
        preds = model_std(images)
        loss = criterion_std(preds, labels)
        loss.backward()
        optimizer_std.step()

        batch_n = images.size(0)
        running_loss += loss.item() * batch_n
        n_seen += batch_n

    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {running_loss / max(1, n_seen):.4f}")

print("\nTraining complete.")

# Evaluate on stratified standards validation split
model_std.eval()
preds_list = []
labels_list = []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        preds = model_std(images)
        preds_list.append(preds.detach().cpu().numpy())
        labels_list.append(labels.numpy())

preds_val = np.concatenate(preds_list)
y_val = np.concatenate(labels_list)
abs_err = np.abs(preds_val - y_val)
mae_val = float(np.mean(abs_err))
w1 = float(np.mean(abs_err <= 1) * 100)
w2 = float(np.mean(abs_err <= 2) * 100)
w3 = float(np.mean(abs_err <= 3) * 100)

print("\nStratified standards validation performance:")
print(f"  MAE: {mae_val:.3f}")
print(f"  Within +/-1: {w1:.1f}%")
print(f"  Within +/-2: {w2:.1f}%")
print(f"  Within +/-3: {w3:.1f}%")

results_std_df = pd.DataFrame({
    "PanelID": groups_val,
    "Prediction": preds_val,
    "TrueLabel": y_val,
})
results_std_df["AbsError"] = np.abs(results_std_df["Prediction"] - results_std_df["TrueLabel"])

print("\nPreview of stratified standards predictions:")
display(results_std_df.head(20))

panel_stats_std = (
    results_std_df
    .groupby("PanelID", dropna=False)
    .agg(
        AvgPrediction=("Prediction", "mean"),
        AvgTrueLabel=("TrueLabel", "mean"),
        MeanAbsError=("AbsError", "mean"),
        N=("Prediction", "count"),
    )
    .reset_index()
    .sort_values("MeanAbsError", ascending=False)
)

print("\nPanel-level stratified standards summary:")
display(panel_stats_std.head(20))

TRAIN ON STANDARDS-TRAIN, TEST ON STRATIFIED STANDARDS-VAL
USING OPTUNA BEST PARAMS
Standards total: 177
Train images: 141 | Val images: 36
Split type: random stratified by label (not panel-safe)
Best params:
  img_size: 92
  num_blocks: 5
  base_channels: 12
  hidden_dim: 256
  dropout: 0.2511572371061875
  use_batchnorm: True
  batch_size: 32
  optimizer: Adam
  lr: 0.0006049716507542575
  weight_decay: 1.963434157293331e-08
  rotation_deg: 6
  crop_scale: 0.9608106745617722
  loss: SmoothL1
Epoch 2/50 - Loss: 2.0743
Epoch 4/50 - Loss: 1.0793
Epoch 6/50 - Loss: 0.7203
Epoch 8/50 - Loss: 0.6752
Epoch 10/50 - Loss: 0.5495
Epoch 12/50 - Loss: 0.6324
Epoch 14/50 - Loss: 0.7702
Epoch 16/50 - Loss: 0.4010
Epoch 18/50 - Loss: 0.9031
Epoch 20/50 - Loss: 0.4660
Epoch 22/50 - Loss: 0.5487
Epoch 24/50 - Loss: 0.5191
Epoch 26/50 - Loss: 0.5545
Epoch 28/50 - Loss: 0.6378
Epoch 30/50 - Loss: 0.6657
Epoch 32/50 - Loss: 0.7124
Epoch 34/50 - Loss: 0.4614
Epoch 36/50 - Loss: 0.4662
Epoch 38/50 - Loss:

,PanelID,Prediction,TrueLabel,AbsError
0,NaN,1.118060,1.0,0.118060
1,NaN,8.593822,9.0,0.406178
2,NaN,2.830659,3.0,0.169341
3,NaN,3.654747,4.0,0.345253
4,NaN,4.251426,5.0,0.748574
5,NaN,3.192967,3.0,0.192967
6,NaN,1.891793,2.0,0.108207
7,NaN,6.028652,7.0,0.971348
8,NaN,7.838217,8.0,0.161783
9,NaN,9.146523,10.0,0.853477



Panel-level stratified standards summary:


,PanelID,AvgPrediction,AvgTrueLabel,MeanAbsError,N
0,NaN,5.115454,5.5,0.464845,36


In [10]:
# lets look at the optuna model parameters but trained for 80 epochs
# so we want train on standards test on real paint, no calibration and also
# report the c-index

In [13]:
# best Optuna params, trained for 80 epochs on standards, tested on real paint (no calibration)
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

if "study" not in globals() or len(study.trials) == 0:
    raise RuntimeError("Run the Optuna optimization cell first so study.best_trial is available.")

if "set_global_seed" not in globals():
    def set_global_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

set_global_seed(SEED)

# Use best Optuna hyperparameters
best_p = study.best_trial.params
img_size = int(best_p["img_size"])
epochs_80 = 80

# Rebuild tensors with best image size
X_std, y_std, groups_std = prepare_tensors(data_standards, img_size=(img_size, img_size))
X_real, y_real, groups_real = prepare_tensors(data_real_paint, img_size=(img_size, img_size))

train_tf, test_tf = make_transforms(
    img_size=img_size,
    rotation_deg=int(best_p["rotation_deg"]),
    crop_scale=float(best_p["crop_scale"]),
)

# Train on ALL standards
std_train_ds = TensorListDataset(X_std, y_std, transform=train_tf)
std_sampler = weighted_sampler_from_y(y_std, seed=SEED)
std_train_loader = DataLoader(
    std_train_ds,
    batch_size=int(best_p["batch_size"]),
    sampler=std_sampler,
    drop_last=False,
)

# Test on real paint (no calibration)
real_test_ds = TensorListDataset(X_real, y_real, transform=test_tf)
real_test_loader = DataLoader(
    real_test_ds,
    batch_size=int(best_p["batch_size"]),
    shuffle=False,
    drop_last=False,
)

model_80 = WaveletCNNRegressor(
    num_blocks=int(best_p["num_blocks"]),
    base_channels=int(best_p["base_channels"]),
    hidden_dim=int(best_p["hidden_dim"]),
    dropout=float(best_p["dropout"]),
    use_batchnorm=bool(best_p["use_batchnorm"]),
).to(DEVICE)

optimizer_80 = make_optimizer(
    best_p["optimizer"],
    model_80.parameters(),
    float(best_p["lr"]),
    float(best_p["weight_decay"]),
)
criterion_80 = make_loss(best_p["loss"])

print("=" * 70)
print("BEST OPTUNA PARAMS | TRAIN: STANDARDS (80 EPOCHS) | TEST: REAL PAINT")
print("NO CALIBRATION")
print("=" * 70)
print(f"Best trial #: {study.best_trial.number}")
print(f"Best trial objective value (standards val MAE): {study.best_trial.value:.4f}")
print("Params:")
for k, v in best_p.items():
    print(f"  {k}: {v}")

for epoch in range(epochs_80):
    model_80.train()
    running_loss = 0.0
    n_seen = 0

    for images, labels in std_train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_80.zero_grad()
        preds = model_80(images)
        loss = criterion_80(preds, labels)
        loss.backward()
        optimizer_80.step()

        batch_n = images.size(0)
        running_loss += loss.item() * batch_n
        n_seen += batch_n

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch + 1}/{epochs_80} - train loss: {running_loss / max(1, n_seen):.4f}")

# Inference on real paint
model_80.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for images, labels in real_test_loader:
        images = images.to(DEVICE)
        preds = model_80(images)
        preds_all.append(preds.detach().cpu().numpy())
        labels_all.append(labels.numpy())

preds_np = np.concatenate(preds_all)
labels_np = np.concatenate(labels_all)
abs_err = np.abs(preds_np - labels_np)

print("\nReal-paint evaluation (raw, no calibration):")
print(f"  MAE: {float(abs_err.mean()):.4f}")
print(f"  Within +/-1: {float((abs_err <= 1).mean() * 100):.2f}%")
print(f"  Within +/-2: {float((abs_err <= 2).mean() * 100):.2f}%")
print(f"  Within +/-3: {float((abs_err <= 3).mean() * 100):.2f}%")
print(f"  Within +/-4: {float((abs_err <= 4).mean() * 100):.2f}%")

results_real_raw_80_df = pd.DataFrame(
    {
        "PanelID": groups_real,
        "Prediction": preds_np,
        "TrueLabel": labels_np,
    }
)
results_real_raw_80_df["Difference"] = results_real_raw_80_df["Prediction"] - results_real_raw_80_df["TrueLabel"]
results_real_raw_80_df["AbsDifference"] = np.abs(results_real_raw_80_df["Difference"])

def concordance_index(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    n = len(y_true)
    if n < 2:
        return np.nan, 0

    concordant = 0.0
    comparable = 0

    for i in range(n - 1):
        dy = y_true[i + 1:] - y_true[i]
        dp = y_pred[i + 1:] - y_pred[i]

        comp_mask = dy != 0
        if not np.any(comp_mask):
            continue

        dy = dy[comp_mask]
        dp = dp[comp_mask]

        comparable += dy.size
        concordant += np.sum((dy * dp) > 0)
        concordant += 0.5 * np.sum(dp == 0)

    if comparable == 0:
        return np.nan, 0

    return concordant / comparable, comparable

# C-index: image-level
c_img, n_img_pairs = concordance_index(
    results_real_raw_80_df["TrueLabel"],
    results_real_raw_80_df["Prediction"],
)

# C-index: panel-level
panel_df = (
    results_real_raw_80_df
    .groupby("PanelID", dropna=False)
    .agg(
        TrueLabel=("TrueLabel", "mean"),
        Pred=("Prediction", "mean"),
    )
    .reset_index()
)
c_panel, n_panel_pairs = concordance_index(panel_df["TrueLabel"], panel_df["Pred"])
print("\nC-index (real paint, no calibration):")
print(f"  Image-level C-index: {c_img:.4f} (comparable pairs: {n_img_pairs})")
print(f"  Panel-level C-index: {c_panel:.4f} (comparable pairs: {n_panel_pairs})")

worst_20 = (
    results_real_raw_80_df
    .sort_values("AbsDifference", ascending=False)
    .head(20)
    .loc[:, ["PanelID", "TrueLabel", "Prediction", "Difference", "AbsDifference"]]
    .reset_index(drop=True)
)

print("\nTop 20 worst predictions (by absolute error):")
display(worst_20)

print("\nPreview:")
display(results_real_raw_80_df.head(20))

BEST OPTUNA PARAMS | TRAIN: STANDARDS (80 EPOCHS) | TEST: REAL PAINT
NO CALIBRATION
Best trial #: 1
Best trial objective value (standards val MAE): 0.4067
Params:
  img_size: 92
  num_blocks: 5
  base_channels: 12
  hidden_dim: 256
  dropout: 0.2511572371061875
  use_batchnorm: True
  batch_size: 32
  optimizer: Adam
  lr: 0.0006049716507542575
  weight_decay: 1.963434157293331e-08
  rotation_deg: 6
  crop_scale: 0.9608106745617722
  loss: SmoothL1
  Epoch 1/80 - train loss: 3.7604
  Epoch 10/80 - train loss: 0.5321
  Epoch 20/80 - train loss: 0.5433
  Epoch 30/80 - train loss: 0.3497
  Epoch 40/80 - train loss: 0.4812
  Epoch 50/80 - train loss: 0.5008
  Epoch 60/80 - train loss: 0.4157
  Epoch 70/80 - train loss: 0.4270
  Epoch 80/80 - train loss: 0.3768

Real-paint evaluation (raw, no calibration):
  MAE: 1.4673
  Within +/-1: 37.44%
  Within +/-2: 74.65%
  Within +/-3: 91.16%
  Within +/-4: 100.00%

C-index (real paint, no calibration):
  Image-level C-index: 0.8565 (comparable pai

,PanelID,TrueLabel,Prediction,Difference,AbsDifference
0,25-124,9.5,5.515914,-3.984086,3.984086
1,25-124,9.5,5.540832,-3.959168,3.959168
2,25-124,9.5,5.544206,-3.955794,3.955794
3,25-154,10.0,6.068808,-3.931192,3.931192
4,25-124,9.5,5.576829,-3.923171,3.923171
5,25-154,10.0,6.094619,-3.905381,3.905381
6,25-124,9.5,5.598589,-3.901411,3.901411
7,25-124,9.5,5.603065,-3.896935,3.896935
8,25-154,10.0,6.149480,-3.850520,3.850520
9,25-154,10.0,6.186108,-3.813892,3.813892



Preview:


,PanelID,Prediction,TrueLabel,Difference,AbsDifference
0,25-130,2.231065,1.3,0.931065,0.931065
1,25-130,2.213568,1.3,0.913568,0.913568
2,25-130,1.884959,1.3,0.584959,0.584959
3,25-130,2.207510,1.3,0.907510,0.907510
4,25-130,2.374882,1.3,1.074882,1.074882
5,25-130,2.254377,1.3,0.954377,0.954377
6,25-101,4.240200,2.2,2.040200,2.040200
7,25-101,3.923323,2.2,1.723323,1.723323
8,25-101,4.177511,2.2,1.977511,1.977511
9,25-101,4.186389,2.2,1.986389,1.986389


In [12]:
# now lets try adding in real paint to the training data, but we only are going to look at
# panels that have a std less than 1 from the HumanRatings.csv file in the std column.

In [13]:
# train on standards + low-std real paint panels; test on held-out low-std real paint panels (panel-safe split)
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

if "data_standards" not in globals() or "data_real_paint" not in globals():
    raise RuntimeError("Run Cells 1-5 first so data_standards and data_real_paint are available.")

if "best_p" not in globals():
    if "study" in globals() and len(study.trials) > 0:
        best_p = study.best_trial.params
    else:
        raise RuntimeError("Run the Optuna cells first so best_p (or study.best_trial.params) is available.")

# -------- Helpers --------
def _normalize_label(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "":
        return None
    m = re.search(r"\d+", s)
    if m is not None:
        return str(int(m.group(0)))
    return s

def _extract_label_from_panel_id(panel_id):
    """Extract the HumanRatings label key from PanelID strings like '25-100'."""
    if pd.isna(panel_id):
        return None
    s = str(panel_id).strip()
    if s == "":
        return None

    if "-" in s:
        tail = s.split("-")[-1].strip()
        m_tail = re.search(r"\d+", tail)
        if m_tail is not None:
            return str(int(m_tail.group(0)))

    all_nums = re.findall(r"\d+", s)
    if all_nums:
        return str(int(all_nums[-1]))
    return s

def _concordance_index(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    n = len(y_true)
    if n < 2:
        return np.nan, 0

    concordant = 0.0
    comparable = 0
    for i in range(n - 1):
        dy = y_true[i + 1:] - y_true[i]
        dp = y_pred[i + 1:] - y_pred[i]

        comp_mask = dy != 0
        if not np.any(comp_mask):
            continue

        dy = dy[comp_mask]
        dp = dp[comp_mask]

        comparable += dy.size
        concordant += np.sum((dy * dp) > 0)
        concordant += 0.5 * np.sum(dp == 0)

    if comparable == 0:
        return np.nan, 0
    return concordant / comparable, comparable

# -------- Read human-rating panel std --------
hr = pd.read_csv("../analysis/HumanRatings.csv")
hr.columns = [c.strip() for c in hr.columns]

if "Label" not in hr.columns or "std" not in hr.columns:
    raise RuntimeError("Expected columns 'Label' and 'std' in ../human_ratings/HumanRatings.csv")

hr["Label_norm"] = hr["Label"].map(_normalize_label)
hr["std"] = pd.to_numeric(hr["std"], errors="coerce")
low_std_label_set = set(hr.loc[hr["std"] < 1, "Label_norm"].dropna().astype(str).tolist())

if len(low_std_label_set) < 2:
    raise RuntimeError("Need at least 2 low-std labels (std < 1) to create train/test split.")

# -------- Build real-paint mapping --------
real_panel_raw = np.array([None if d.get("PanelID", None) is None else str(d.get("PanelID", None)).strip() for d in data_real_paint], dtype=object)
real_label_from_panel = np.array([_extract_label_from_panel_id(p) for p in real_panel_raw], dtype=object)
real_idx_all = np.arange(len(data_real_paint))

is_low_std_real = np.array([lbl in low_std_label_set for lbl in real_label_from_panel], dtype=bool)
low_std_real_idx = real_idx_all[is_low_std_real]

if len(low_std_real_idx) == 0:
    sample_labels = sorted(set([x for x in real_label_from_panel if x is not None]))[:15]
    raise RuntimeError(
        "No real-paint images matched low-std labels. "
        f"Example labels parsed from real PanelID: {sample_labels}"
    )

# Split by full PanelID (leakage-safe)
eligible_panels = sorted(set([p for p in real_panel_raw[low_std_real_idx] if p is not None and p != ""]))
if len(eligible_panels) < 2:
    raise RuntimeError("Need at least 2 eligible low-std real-paint PanelID groups for panel-safe split.")

rng = np.random.default_rng(42)
eligible_panels = np.array(eligible_panels, dtype=object)
rng.shuffle(eligible_panels)

real_train_fraction = 0.7
n_train_panels = int(round(real_train_fraction * len(eligible_panels)))
n_train_panels = max(1, min(len(eligible_panels) - 1, n_train_panels))

train_real_panels = set(eligible_panels[:n_train_panels].tolist())
test_real_panels = set(eligible_panels[n_train_panels:].tolist())

train_real_idx = np.array([i for i in low_std_real_idx if real_panel_raw[i] in train_real_panels], dtype=int)
test_real_idx = np.array([i for i in low_std_real_idx if real_panel_raw[i] in test_real_panels], dtype=int)

if len(train_real_idx) == 0 or len(test_real_idx) == 0:
    raise RuntimeError("Panel split produced empty train or test set. Adjust split fraction.")

# -------- Prepare tensors --------
img_size = int(best_p["img_size"])
X_std, y_std, _ = prepare_tensors(data_standards, img_size=(img_size, img_size))
X_real_all, y_real_all, groups_real_all = prepare_tensors(data_real_paint, img_size=(img_size, img_size))

# Standards + low-std real-paint train panels
X_train_mix = list(X_std) + [X_real_all[i] for i in train_real_idx]
y_train_mix = np.concatenate([y_std, y_real_all[train_real_idx].astype(np.float32)])

# Held-out low-std real-paint test panels
X_test_holdout = [X_real_all[i] for i in test_real_idx]
y_test_holdout = y_real_all[test_real_idx].astype(np.float32)
groups_test_holdout = groups_real_all[test_real_idx]

train_tf, test_tf = make_transforms(
    img_size=img_size,
    rotation_deg=int(best_p["rotation_deg"]),
    crop_scale=float(best_p["crop_scale"]),
)

train_ds = TensorListDataset(X_train_mix, y_train_mix, transform=train_tf)
test_ds = TensorListDataset(X_test_holdout, y_test_holdout, transform=test_tf)

train_sampler = weighted_sampler_from_y(y_train_mix)
train_loader = DataLoader(
    train_ds,
    batch_size=int(best_p["batch_size"]),
    sampler=train_sampler,
    drop_last=False,
)
test_loader = DataLoader(
    test_ds,
    batch_size=int(best_p["batch_size"]),
    shuffle=False,
    drop_last=False,
)

model_mix = WaveletCNNRegressor(
    num_blocks=int(best_p["num_blocks"]),
    base_channels=int(best_p["base_channels"]),
    hidden_dim=int(best_p["hidden_dim"]),
    dropout=float(best_p["dropout"]),
    use_batchnorm=bool(best_p["use_batchnorm"]),
).to(DEVICE)

optimizer_mix = make_optimizer(
    best_p["optimizer"],
    model_mix.parameters(),
    float(best_p["lr"]),
    float(best_p["weight_decay"]),
)
criterion_mix = make_loss(best_p["loss"])
epochs_mix = 60

print("=" * 78)
print("TRAIN: standards + low-std real-paint(train panels) | TEST: low-std real-paint(test panels)")
print("Panel-safe split by full PanelID (no leakage)")
print("=" * 78)
print("Low-std filter source: HumanRatings std < 1")
print(f"Low-std labels in HumanRatings: {len(low_std_label_set)}")
print(f"Matched real-paint low-std images: {len(low_std_real_idx)}")
print(f"Eligible real-paint PanelIDs: {len(eligible_panels)}")
print(f"Real-paint PanelID split -> train: {len(train_real_panels)} | test: {len(test_real_panels)}")
print(f"Image counts -> standards train: {len(X_std)}, real train: {len(train_real_idx)}, real test: {len(test_real_idx)}")

for epoch in range(epochs_mix):
    model_mix.train()
    running_loss = 0.0
    n_seen = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_mix.zero_grad()
        preds = model_mix(images)
        loss = criterion_mix(preds, labels)
        loss.backward()
        optimizer_mix.step()

        batch_n = images.size(0)
        running_loss += loss.item() * batch_n
        n_seen += batch_n

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch + 1}/{epochs_mix} - train loss: {running_loss / max(1, n_seen):.4f}")

# -------- Evaluate on held-out low-std real paint --------
model_mix.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        preds = model_mix(images)
        preds_all.append(preds.detach().cpu().numpy())
        labels_all.append(labels.numpy())

preds_np = np.concatenate(preds_all)
labels_np = np.concatenate(labels_all)
abs_err = np.abs(preds_np - labels_np)

results_lowstd_holdout_df = pd.DataFrame(
    {
        "PanelID": groups_test_holdout,
        "Prediction": preds_np,
        "TrueLabel": labels_np,
    }
)
results_lowstd_holdout_df["Difference"] = results_lowstd_holdout_df["Prediction"] - results_lowstd_holdout_df["TrueLabel"]
results_lowstd_holdout_df["AbsDifference"] = np.abs(results_lowstd_holdout_df["Difference"])
print("\nHeld-out low-std real-paint performance:")
print(f"  MAE: {float(abs_err.mean()):.4f}")
print(f"  Within +/-1: {float((abs_err <= 1).mean() * 100):.2f}%")
print(f"  Within +/-2: {float((abs_err <= 2).mean() * 100):.2f}%")
print(f"  Within +/-3: {float((abs_err <= 3).mean() * 100):.2f}%")
print(f"  Within +/-4: {float((abs_err <= 4).mean() * 100):.2f}%")

c_img, n_img_pairs = _concordance_index(
    results_lowstd_holdout_df["TrueLabel"],
    results_lowstd_holdout_df["Prediction"],
)

panel_df = (
    results_lowstd_holdout_df
    .groupby("PanelID", dropna=False)
    .agg(TrueLabel=("TrueLabel", "mean"), Pred=("Prediction", "mean"))
    .reset_index()
)
c_panel, n_panel_pairs = _concordance_index(panel_df["TrueLabel"], panel_df["Pred"])
print("\nC-index (held-out low-std real paint):")
print(f"  Image-level C-index: {c_img:.4f} (comparable pairs: {n_img_pairs})")
print(f"  Panel-level C-index: {c_panel:.4f} (comparable pairs: {n_panel_pairs})")

worst_20 = (
    results_lowstd_holdout_df
    .sort_values("AbsDifference", ascending=False)
    .head(20)
    .loc[:, ["PanelID", "TrueLabel", "Prediction", "Difference", "AbsDifference"]]
    .reset_index(drop=True)
)

print("\nTop 20 worst predictions (held-out low-std real paint):")
display(worst_20)

print("\nPreview:")
display(results_lowstd_holdout_df.head(20))

TRAIN: standards + low-std real-paint(train panels) | TEST: low-std real-paint(test panels)
Panel-safe split by full PanelID (no leakage)
Low-std filter source: HumanRatings std < 1
Low-std labels in HumanRatings: 32
Matched real-paint low-std images: 257
Eligible real-paint PanelIDs: 32
Real-paint PanelID split -> train: 22 | test: 10
Image counts -> standards train: 177, real train: 169, real test: 88
  Epoch 1/60 - train loss: 2.1582
  Epoch 10/60 - train loss: 0.7031
  Epoch 20/60 - train loss: 0.6534


KeyboardInterrupt: 

In [ ]:
#
# insight: optuna model trained with only standards did pretty much the same as model
# trained with standards + low-std real paint, so lets go with the model trained only on the standards
# also visually inspected the top worst panels the model was predicting and based on the gel site images
# of the standards and the real paint image, I agree much more with the model
# than the human ratings.
#

In [ ]:
# best current model

In [ ]:
# hyperparameter values:

In [ ]:
# best model parameters I had before 
"""
Params:
  img_size: 92
  num_blocks: 5
  base_channels: 24
  hidden_dim: 128
  dropout: 0.48184968246925675
  use_batchnorm: True
  batch_size: 16
  optimizer: Adam
  lr: 0.00047109025136420144
  weight_decay: 1.3667272915456171e-08
  rotation_deg: 2
  crop_scale: 0.9572074102782637
  loss: SmoothL1

"""

In [21]:
# model with fixed wavelet parameters: train on standards, evaluate on real paint, save final weights each run
import os
import random
from datetime import datetime
import numpy as np
import torch
import pandas as pd
from torch.utils.data import DataLoader

# Ensure required objects from earlier cells exist
required_names = [
    "data_standards",
    "data_real_paint",
    "prepare_tensors",
    "make_transforms",
    "TensorListDataset",
    "weighted_sampler_from_y",
    "WaveletCNNRegressor",
    "make_optimizer",
    "make_loss",
    "DEVICE",
]
missing = [name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(f"Run earlier setup cells first. Missing: {missing}")

# Randomize initialization every run (different seed each execution)
run_seed = 3
random.seed(run_seed)
np.random.seed(run_seed)
torch.manual_seed(run_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(run_seed)

# Fixed parameters you provided
fixed_p = {
    "img_size": 92,
    "num_blocks": 5,
    "base_channels": 24,
    "hidden_dim": 128,
    "dropout": 0.48184968246925675,
    "use_batchnorm": True,
    "batch_size": 16,
    "optimizer": "Adam",
    "lr": 0.00047109025136420144,
    "weight_decay": 1.3667272915456171e-08,
    "rotation_deg": 2,
    "crop_scale": 0.9572074102782637,
    "loss": "SmoothL1",
}
epochs = 70

img_size = int(fixed_p["img_size"])
X_std, y_std, groups_std = prepare_tensors(data_standards, img_size=(img_size, img_size))
X_real, y_real, groups_real = prepare_tensors(data_real_paint, img_size=(img_size, img_size))

train_tf, test_tf = make_transforms(
    img_size=img_size,
    rotation_deg=int(fixed_p["rotation_deg"]),
    crop_scale=float(fixed_p["crop_scale"]),
)

# Train on all standards
std_train_ds = TensorListDataset(X_std, y_std, transform=train_tf)
std_sampler = weighted_sampler_from_y(y_std, seed=run_seed)
std_train_loader = DataLoader(
    std_train_ds,
    batch_size=int(fixed_p["batch_size"]),
    sampler=std_sampler,
    drop_last=False,
)

# Evaluate on all real paint
real_test_ds = TensorListDataset(X_real, y_real, transform=test_tf)
real_test_loader = DataLoader(
    real_test_ds,
    batch_size=int(fixed_p["batch_size"]),
    shuffle=False,
    drop_last=False,
)

model_fixed = WaveletCNNRegressor(
    num_blocks=int(fixed_p["num_blocks"]),
    base_channels=int(fixed_p["base_channels"]),
    hidden_dim=int(fixed_p["hidden_dim"]),
    dropout=float(fixed_p["dropout"]),
    use_batchnorm=bool(fixed_p["use_batchnorm"]),
).to(DEVICE)

optimizer_fixed = make_optimizer(
    fixed_p["optimizer"],
    model_fixed.parameters(),
    float(fixed_p["lr"]),
    float(fixed_p["weight_decay"]),
)
criterion_fixed = make_loss(fixed_p["loss"])

print("=" * 76)
print("FIXED-PARAM WAVELET MODEL | TRAIN: STANDARDS (70 EPOCHS) | TEST: REAL PAINT")
print("Random initialization enabled: new seed per run")
print("=" * 76)
print(f"Run seed: {run_seed}")
print(f"Standards train size: {len(X_std)} | Real-paint test size: {len(X_real)}")

for epoch in range(epochs):
    model_fixed.train()
    running_loss = 0.0
    n_seen = 0

    for images, labels in std_train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer_fixed.zero_grad()
        preds = model_fixed(images)
        loss = criterion_fixed(preds, labels)
        loss.backward()
        optimizer_fixed.step()

        batch_n = images.size(0)
        running_loss += loss.item() * batch_n
        n_seen += batch_n

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch + 1}/{epochs} - train loss: {running_loss / max(1, n_seen):.4f}")

# Inference on real paint
model_fixed.eval()
preds_all, labels_all = [], []
with torch.no_grad():
    for images, labels in real_test_loader:
        images = images.to(DEVICE)
        preds = model_fixed(images)
        preds_all.append(preds.detach().cpu().numpy())
        labels_all.append(labels.numpy())

preds_np = np.concatenate(preds_all)
labels_np = np.concatenate(labels_all)
abs_err = np.abs(preds_np - labels_np)

print("\nReal-paint evaluation (raw, no calibration):")
print(f"  MAE: {float(abs_err.mean()):.4f}")
print(f"  Within +/-1: {float((abs_err <= 1).mean() * 100):.2f}%")
print(f"  Within +/-2: {float((abs_err <= 2).mean() * 100):.2f}%")
print(f"  Within +/-3: {float((abs_err <= 3).mean() * 100):.2f}%")
print(f"  Within +/-4: {float((abs_err <= 4).mean() * 100):.2f}%")

results_real_fixed_df = pd.DataFrame(
    {
        "PanelID": groups_real,
        "Prediction": preds_np,
        "TrueLabel": labels_np,
    }
)
results_real_fixed_df["Difference"] = results_real_fixed_df["Prediction"] - results_real_fixed_df["TrueLabel"]
results_real_fixed_df["AbsDifference"] = np.abs(results_real_fixed_df["Difference"])

print("\nPreview:")
display(results_real_fixed_df.head(20))

# Save final model weights and run metadata
os.makedirs("../models", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f"../models/wavelet_fixed_standards_to_real_e80_seed{run_seed}_{timestamp}.pt"

checkpoint = {
    "model_state_dict": model_fixed.state_dict(),
    "params": fixed_p,
    "epochs": epochs,
    "run_seed": run_seed,
    "metrics": {
        "mae": float(abs_err.mean()),
        "within_1": float((abs_err <= 1).mean() * 100),
        "within_2": float((abs_err <= 2).mean() * 100),
        "within_3": float((abs_err <= 3).mean() * 100),
        "within_4": float((abs_err <= 4).mean() * 100),
    },
}
torch.save(checkpoint, save_path)
print(f"\nSaved final model checkpoint to: {save_path}")

FIXED-PARAM WAVELET MODEL | TRAIN: STANDARDS (80 EPOCHS) | TEST: REAL PAINT
Random initialization enabled: new seed per run
Run seed: 3
Standards train size: 177 | Real-paint test size: 430
  Epoch 1/70 - train loss: 2.3704
  Epoch 10/70 - train loss: 0.8189
  Epoch 20/70 - train loss: 0.7794
  Epoch 30/70 - train loss: 0.6744
  Epoch 40/70 - train loss: 0.8609
  Epoch 50/70 - train loss: 0.7848
  Epoch 60/70 - train loss: 0.6258
  Epoch 70/70 - train loss: 0.7196

Real-paint evaluation (raw, no calibration):
  MAE: 0.6205
  Within +/-1: 79.30%
  Within +/-2: 98.37%
  Within +/-3: 99.53%
  Within +/-4: 100.00%

Preview:


,PanelID,Prediction,TrueLabel,Difference,AbsDifference
0,25-130,1.482275,1.3,0.182275,0.182275
1,25-130,1.219151,1.3,-0.080849,0.080849
2,25-130,1.318456,1.3,0.018456,0.018456
3,25-130,1.251817,1.3,-0.048183,0.048183
4,25-130,1.396792,1.3,0.096792,0.096792
5,25-130,1.629585,1.3,0.329585,0.329585
6,25-101,3.074586,2.2,0.874586,0.874586
7,25-101,2.563390,2.2,0.363390,0.363390
8,25-101,2.782194,2.2,0.582194,0.582194
9,25-101,2.329475,2.2,0.129475,0.129475



Saved final model checkpoint to: ../models/wavelet_fixed_standards_to_real_e80_seed3_20260729_151845.pt


In [ ]:
"""
required_names = [
    "data_standards",
    "data_real_paint",
    "prepare_tensors",
    "make_transforms",
    "TensorListDataset",
    "weighted_sampler_from_y",
    "WaveletCNNRegressor",
    "make_optimizer",
    "make_loss",
    "DEVICE",
"""